In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

# Install dependencies

In [ ]:
!pip install mplfinance --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 1.6 MB/s eta 0:00:00


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Configuration

In [ ]:
import os

# ⚠️ CHANGE THIS FOR EACH COLAB ACCOUNT:
# Account 1: (0, 20)
# Account 2: (20, 40)
# Account 3: (40, 60)
# Account 4: (60, 80)
ACCOUNT_RANGE = (3, 20)  # <-- CHANGE THIS

# Paths
CSV_PATH  = '/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv'
OUTPUT_DIR = '/content/drive/MyDrive/DMIF/charts_final'
LOCAL_CSV  = '/content/master_dataset.csv'

# Chart settings
SEQ_LEN  = 30
STRIDE   = 5
IMG_SIZE = 224

# ── FIXED column names to match your actual CSV ──────────────
HEATMAP_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Share_Volume',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
    'MA_24', 'MA_30', 'SAR', 'Fib_0618'
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Account range: companies {ACCOUNT_RANGE[0]} to {ACCOUNT_RANGE[1]}")
print(f"HEATMAP_FEATURES: {HEATMAP_FEATURES}")

Account range: companies 0 to 20
HEATMAP_FEATURES: ['Open', 'High', 'Low', 'Close', 'Share_Volume', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'MA_24', 'MA_30', 'SAR', 'Fib_0618']


# Copy CSV to local disk

In [ ]:
import shutil
print("Copying CSV to local disk...")
shutil.copy(CSV_PATH, LOCAL_CSV)
print("Done.")


Copying CSV to local disk...
Done.


# Load data and get company list

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(LOCAL_CSV, parse_dates=['Date'])
df = df.sort_values(['Company_Code', 'Date']).reset_index(drop=True)

# Get all unique companies sorted alphabetically by symbol
all_companies = sorted(df['Company_Code'].unique())
print(f"Total companies in dataset: {len(all_companies)}")

# Select this account's slice
start_idx, end_idx = ACCOUNT_RANGE
my_companies = all_companies[start_idx:end_idx]
print(f"This account will process: {my_companies}")

Total companies in dataset: 80
This account will process: ['AAF.N', 'AAIC.N', 'ABAN.N', 'ABL.N', 'ACAP.N', 'ACME.N', 'AFS.N', 'AGAL.N', 'AGPL.N', 'AGST.N', 'AGST.X', 'AHPL.N', 'AHUN.N', 'AINS.N', 'ALLI.N', 'ALUM.N', 'AMF.N', 'AMSL.N', 'APLA.N', 'ASCO.N']


In [ ]:
print("Available columns:")
print(df.columns.tolist())

print("\nChecking HEATMAP_FEATURES availability:")
for f in HEATMAP_FEATURES:
    exists = f in df.columns
    print(f"  {f:<20} {'✓' if exists else '✗ MISSING'}")

Available columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Trade_Volume', 'Share_Volume', 'Turnover', 'MA_24', 'MA_30', 'MA_500', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width', 'SAR', 'SAR_Trend', 'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786', 'Volume_MA_10', 'Volume_Ratio', 'Log_Volume', 'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range', 'Volatility_10', 'W_High', 'W_Low', 'W_Close', 'W_Volume', 'W_Turnover', 'W_Days', 'W_RSI', 'W_MACD', 'W_MA_10', 'W_MA_20', 'W_Return', 'W_Volatility', 'M_High', 'M_Low', 'M_Close', 'M_Volume', 'M_Turnover', 'M_Days', 'M_RSI', 'M_MACD', 'M_MA_6', 'M_MA_12', 'M_Return', 'M_Volatility', 'Q_High', 'Q_Low', 'Q_Close', 'Q_Volume', 'Q_Turnover', 'Q_Days', 'Q_RSI', 'Q_MACD', 'Q_MA_4', 'Q_MA_8', 'Q_Return', 'Q_Volatility', 'Y_High', 'Y_Low', 'Y_Close', 'Y_Volume', 'Y_Turnover', 'Y_Days', 'Y_RSI', 'Y_MACD', 'Y_MA_3', 'Y_MA_5', 'Y_Return', 'Y_Volatility', 'Company_Code', 'Target']

Checking HEATMAP_FE

# Chart generation functions

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless, no display needed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

def generate_candlestick(window_df, save_path):
    """
    Candlestick chart with volume panel.
    Bullish = teal, Bearish = red, dark background.
    As described in paper Section 5.4.2
    """
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(IMG_SIZE/100, IMG_SIZE/100),
        gridspec_kw={'height_ratios': [3, 1]},
        dpi=100
    )
    fig.patch.set_facecolor('#0d0d0d')
    ax1.set_facecolor('#0d0d0d')
    ax2.set_facecolor('#0d0d0d')

    for i, (_, row) in enumerate(window_df.iterrows()):
        o, h, l, c = row['Open'], row['High'], row['Low'], row['Close']
        color = '#00b4b4' if c >= o else '#ff3333'  # teal or red

        # Wick
        ax1.plot([i, i], [l, h], color=color, linewidth=0.8)
        # Body
        body_bottom = min(o, c)
        body_height = abs(c - o) if abs(c - o) > 0 else 0.001
        ax1.bar(i, body_height, bottom=body_bottom,
                color=color, width=0.6, linewidth=0)

    # Volume bars
    for i, (_, row) in enumerate(window_df.iterrows()):
        color = '#00b4b4' if row['Close'] >= row['Open'] else '#ff3333'
        ax2.bar(i, row['Share_Volume'], color=color, width=0.6,
                linewidth=0, alpha=0.7)

    # Clean up axes
    for ax in [ax1, ax2]:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout(pad=0)
    plt.savefig(save_path, dpi=100,
                bbox_inches='tight',
                facecolor='#0d0d0d',
                pad_inches=0)
    plt.close(fig)


def generate_heatmap(window_df, save_path):
    """
    Pearson correlation heatmap of 14 features over 30-day window.
    Red-yellow-green colormap as described in paper Section 5.4.2
    """
    # Use only available features from the 14
    available = [f for f in HEATMAP_FEATURES if f in window_df.columns]
    corr_data = window_df[available].copy()

    # Drop columns with zero variance (causes NaN in correlation)
    corr_data = corr_data.loc[:, corr_data.std() > 0]

    if corr_data.shape[1] < 2:
        # Not enough features — save blank image
        fig, ax = plt.subplots(figsize=(IMG_SIZE/100, IMG_SIZE/100), dpi=100)
        ax.set_facecolor('#0d0d0d')
        fig.patch.set_facecolor('#0d0d0d')
        plt.savefig(save_path, dpi=100, bbox_inches='tight',
                    facecolor='#0d0d0d', pad_inches=0)
        plt.close(fig)
        return

    corr_matrix = corr_data.corr()

    fig, ax = plt.subplots(
        figsize=(IMG_SIZE/100, IMG_SIZE/100), dpi=100
    )
    fig.patch.set_facecolor('#0d0d0d')

    sns.heatmap(
        corr_matrix,
        ax=ax,
        cmap='RdYlGn',        # red-yellow-green as in paper
        vmin=-1, vmax=1,
        annot=False,
        linewidths=0,
        cbar=False,           # no colorbar to save space
        square=True
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

    plt.tight_layout(pad=0)
    plt.savefig(save_path, dpi=100,
                bbox_inches='tight',
                facecolor='#0d0d0d',
                pad_inches=0)
    plt.close(fig)


# Main generation loop

In [ ]:
import gc
import matplotlib.pyplot as plt

total_generated = 0
total_skipped   = 0
total_errors    = 0

for company in my_companies:
    company_df = df[df['Company_Code'] == company].copy()
    company_df = company_df.sort_values('Date').reset_index(drop=True)

    if len(company_df) < SEQ_LEN + 1:
        print(f"  Skipping {company} — not enough data ({len(company_df)} rows)")
        continue

    # Company output folder
    company_dir = os.path.join(OUTPUT_DIR, company)
    os.makedirs(company_dir, exist_ok=True)

    # Build target label column
    company_df['Target'] = (
        company_df['Close'].shift(-1) > company_df['Close']
    ).astype(float)

    # Slide window
    max_start = len(company_df) - SEQ_LEN - 1
    windows_generated = 0
    company_errors    = 0

    for start in range(0, max_start, STRIDE):
        end       = start + SEQ_LEN
        window_df = company_df.iloc[start:end].copy()
        label     = company_df.iloc[end]['Target']

        if pd.isna(label):
            continue

        label_str = 'UP' if label == 1.0 else 'DOWN'
        pred_date = company_df.iloc[end]['Date'].strftime('%Y%m%d')

        # Build filenames
        cs_filename = f"{company}_candlestick_{pred_date}_{label_str}.png"
        hm_filename = f"{company}_heatmap_{pred_date}_{label_str}.png"
        cs_path     = os.path.join(company_dir, cs_filename)
        hm_path     = os.path.join(company_dir, hm_filename)

        # Skip if both already exist
        cs_exists = os.path.exists(cs_path)
        hm_exists = os.path.exists(hm_path)

        if cs_exists and hm_exists:
            total_skipped += 2
            continue

        # Generate candlestick
        if not cs_exists:
            try:
                generate_candlestick(window_df, cs_path)
                total_generated += 1
            except Exception as e:
                total_errors  += 1
                company_errors += 1
                # Print first 2 errors per company to diagnose
                if company_errors <= 2:
                    print(f"  ✗ Candlestick error [{company}]: {e}")
                    import traceback
                    traceback.print_exc()

        # Generate heatmap
        if not hm_exists:
            try:
                generate_heatmap(window_df, hm_path)
                total_generated += 1
            except Exception as e:
                total_errors   += 1
                company_errors += 1
                if company_errors <= 2:
                    print(f"  ✗ Heatmap error [{company}]: {e}")

        windows_generated += 1

    # Memory cleanup after each company
    plt.close('all')
    gc.collect()

    print(f"  ✓ {company}: {windows_generated} new | "
          f"{company_errors} errors")

print(f"\n{'='*50}")
print(f"Generated : {total_generated} images")
print(f"Skipped   : {total_skipped} (already existed)")
print(f"Errors    : {total_errors}")
print(f"{'='*50}")

  ✓ AAF.N: 642 new | 0 errors
  ✓ AAIC.N: 940 new | 0 errors
  ✓ ABAN.N: 908 new | 0 errors
  ✓ ABL.N: 557 new | 0 errors
  ✓ ACAP.N: 1390 new | 0 errors
  ✓ ACME.N: 1260 new | 0 errors
  ✓ AFS.N: 112 new | 0 errors


# Verification — count images per company

In [ ]:
print("\nVerification — images per company:")
for company in my_companies:
    company_dir = os.path.join(OUTPUT_DIR, company)
    if os.path.exists(company_dir):
        count = len([f for f in os.listdir(company_dir)
                     if f.endswith('.png')])
        print(f"  {company:<15} {count} images")
    else:
        print(f"  {company:<15} folder not found")



Verification — images per company:
  APLA.N          924 images
  ASCO.N          1252 images
